In [ ]:
from jupyter_client.manager import start_new_kernel
import queue, json
from IPython.display import display, SVG, HTML    
km, kc = start_new_kernel(kernel_name="sysml")
#https://github.com/Systems-Modeling/SysML-v2-API-Python-Client/tree/master
#SYSML_VERSION="0.58.0"
#conda install "jupyter-sysml-kernel=$SYSML_VERSION" python=3.* jupyterlab=4.* graphviz=2.* nodejs -c conda-forge -y

In [ ]:
def run(code, timeout=60):
    msg_id = kc.execute(code)
    outputs = []
    while True:
        msg = kc.get_iopub_msg(timeout=timeout)
        if msg["parent_header"].get("msg_id") != msg_id:
            continue
        t = msg["msg_type"]
        if t == "stream":
            outputs.append(msg["content"]["text"])
        elif t in ("execute_result", "display_data"):
            outputs.append(msg["content"]["data"])
        elif t == "error":
            raise RuntimeError("\n".join(msg["content"]["traceback"]))
        elif t == "status" and msg["content"]["execution_state"] == "idle":
            break
    return outputs


In [ ]:
# 1. Your Python-generated SysMLv2 textual syntax:
sysml_src = """
package MyPkg {
    part def Vehicle;
    part myCar : Vehicle;
}
"""

# 2. Parse it in the SysML kernel
run(sysml_src)

In [ ]:
out = run("%help")
print(out[0]['text/plain'])

In [ ]:
run("%repo http://localhost:9000")

In [ ]:
run("%projects")

In [ ]:
run("%publish MyPkg")

In [ ]:
run("%viz MyPkg")

In [ ]:
from IPython.display import display, SVG                                                                                             

def run_viz(code, timeout=60):                                                                                                       
    msg_id = kc.execute(code)                                                                                                      
    while True:                                                                                                                      
        msg = kc.get_iopub_msg(timeout=timeout)                                                                                    
        if msg["parent_header"].get("msg_id") != msg_id:                                                                             
            continue                                                                                                                 
        t = msg["msg_type"]                                                                                                          
        if t in ("execute_result", "display_data"):                                                                                  
            data = msg["content"]["data"]                                                                                            
            if "image/svg+xml" in data:
              display(SVG(data["image/svg+xml"]))                                                                                  
            elif "text/html" in data:                                                                                                
              display(HTML(data["text/html"]))
        elif t == "error":                                                                                                           
            raise RuntimeError("\n".join(msg["content"]["traceback"]))                                                             
        elif t == "status" and msg["content"]["execution_state"] == "idle":                                                          
            break
                                                                                                                                   
run_viz("%viz MyPkg")                                                                                                              

Or, if you want to keep using the original run() and just render whatever it returns:                                                

                                                                                   
                                                                                                                                 


In [ ]:
def render(outputs):
    for o in outputs:
        if isinstance(o, dict):                                                                                                      
            if "image/svg+xml" in o:
                display(SVG(o["image/svg+xml"]))                                                                                     
            elif "text/html" in o:                                                                                                 
                display(HTML(o["text/html"]))                                                                                        
            elif "text/plain" in o:
                print(o["text/plain"])                                                                                               
                                                                                                                                 
render(run("%viz MyPkg"))

In [ ]:
# 3a. Commit to the SysMLv2 API repository (publishes all root elements):
#     %publish takes a project name; the API URL is configured via
#     the PUBLISH_API_BASE_PATH env var (default http://sysml2-dev.intercax.com:9000)
run("%publish MyPkg")

# 3b. OR export the JSON locally instead of publishing:
out = run("%export MyPkg")
# %export returns a JSON element tree you can inspect / POST yourself

In [ ]:
kc.stop_channels()
km.shutdown_kernel()